# Notebook 3: Teacher Model Training

**What this does (in simple terms):**

This is where the "smart" model (teacher) learns to tell real audio
from fake audio. It uses two parts:

- **Wav2Vec2-XLSR** (front-end): A huge pretrained model that "listens"
  to raw audio and extracts meaningful patterns — like how your brain
  processes sound into recognizable features.

- **AASIST** (back-end): A graph attention network that looks at those
  patterns from both the frequency angle (pitch, tone) and time angle
  (rhythm, pauses) to find spoofing clues. Then it decides: real or fake.

The teacher is the big, powerful model. Later in Notebook 4, it will
teach a smaller, faster model (student) what it learned.

**Checkpoint strategy:** The model is saved after every epoch to Google
Drive. If Colab disconnects, you can restart and it will pick up
exactly where it left off — no lost training.

**Run on Colab Pro with GPU. Training takes ~8-15 hours.**

## 3.1 Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers -q

In [ ]:
import os, json, random, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model
from sklearn.metrics import accuracy_score, roc_curve
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
PREPROCESSED_DIR = Path("/content/preprocessed")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/deepfake_project/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000
MAX_DURATION = 6
MAX_SAMPLES = TARGET_SR * MAX_DURATION  # 96000

BATCH_SIZE = 8
NUM_EPOCHS = 10
LEARNING_RATE_W2V = 1e-5       # Low LR for pretrained weights
LEARNING_RATE_BACKEND = 1e-4    # Higher LR for new backend
WEIGHT_DECAY = 1e-4

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Verify preprocessed data exists
assert PREPROCESSED_DIR.exists(), "Preprocessed data not found! Run Notebook 1 first."
for split in ["train", "dev"]:
    count = len(list((PREPROCESSED_DIR / split).glob("*.flac")))
    print(f"{split}: {count} files")

## 3.2 Load Protocols & Build Dataset

Same Dataset code from Notebook 2, included here so this
notebook is self-contained.

In [ ]:
def load_protocol(path):
    df = pd.read_csv(path, sep=" ", header=None,
                     names=["speaker_id", "audio_id", "system_id", "attack_type", "label"])
    df["label_int"] = (df["label"] == "spoof").astype(int)
    return df

PROTOCOL_DIR = PREPROCESSED_DIR / "protocols"
train_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.train.trn.txt")
dev_protocol = load_protocol(PROTOCOL_DIR / "ASVspoof2019.LA.cm.dev.trl.txt")
metadata = pd.read_csv(PREPROCESSED_DIR / "metadata.csv")

print(f"Train: {len(train_protocol)} | Dev: {len(dev_protocol)}")

In [ ]:
# --- Augmentation functions ---
def rawboost_convolutive(audio_np, sr=TARGET_SR):
    n = len(audio_np); spectrum = np.fft.rfft(audio_np)
    freqs = np.fft.rfftfreq(n, d=1.0/sr)
    for _ in range(np.random.randint(1, 5)):
        f = np.random.uniform(100, sr//2-100); bw = np.random.uniform(50, 300)
        spectrum *= 1.0 - np.exp(-0.5*((freqs-f)/(bw/2+1e-8))**2)
    return np.fft.irfft(spectrum, n=n).astype(np.float32)

def rawboost_impulsive(audio_np):
    nl = np.random.uniform(0.001, 0.01)
    return (audio_np + nl*np.random.randn(len(audio_np))*np.abs(audio_np)).astype(np.float32)

def rawboost_augment(a):
    a = a.copy()
    if np.random.rand() < 0.5: a = rawboost_convolutive(a)
    if np.random.rand() < 0.5: a = rawboost_impulsive(a)
    return a

def codec_augment(a, sr=TARGET_SR):
    t = torch.from_numpy(a).unsqueeze(0).float()
    if np.random.rand() < 0.5:
        t = torchaudio.functional.mu_law_decoding(
            torchaudio.functional.mu_law_encoding(t, 256), 256)
    else:
        t = torchaudio.transforms.Resample(sr, 8000)(t)
        t = torchaudio.transforms.Resample(8000, sr)(t)
    return t.squeeze(0).numpy()

def apply_augmentation(audio_tensor):
    a = audio_tensor.numpy().copy()
    if np.random.rand() < 0.7: a = rawboost_augment(a)
    if np.random.rand() < 0.3: a = codec_augment(a)
    return torch.from_numpy(a).float()

In [ ]:
# --- Dataset class ---
class ASVspoofDataset(Dataset):
    def __init__(self, protocol_df, audio_dir, metadata_df, is_train=False):
        self.protocol = protocol_df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.is_train = is_train
        self.length_lookup = {}
        for _, row in metadata_df.iterrows():
            if row.get("status") == "success":
                self.length_lookup[row["audio_id"]] = int(row["original_length"])

    def __len__(self): return len(self.protocol)

    def __getitem__(self, idx):
        row = self.protocol.iloc[idx]
        audio, _ = torchaudio.load(str(self.audio_dir / f"{row['audio_id']}.flac"))
        audio = audio.squeeze(0)
        if self.is_train:
            audio = apply_augmentation(audio)
        mx = torch.max(torch.abs(audio))
        if mx > 0: audio = audio / mx
        orig_len = self.length_lookup.get(row["audio_id"], MAX_SAMPLES)
        mask = torch.zeros(MAX_SAMPLES); mask[:orig_len] = 1.0
        return {"audio": audio, "mask": mask,
                "label": torch.tensor(row["label_int"], dtype=torch.long),
                "audio_id": row["audio_id"], "attack_type": row["attack_type"]}

In [ ]:
train_dataset = ASVspoofDataset(train_protocol, PREPROCESSED_DIR/"train", metadata, is_train=True)
dev_dataset = ASVspoofDataset(dev_protocol, PREPROCESSED_DIR/"dev", metadata, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Train: {len(train_loader)} batches | Dev: {len(dev_loader)} batches")

## 3.3 Model Architecture

In [ ]:
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a_src = nn.Linear(out_dim, 1, bias=False)
        self.a_dst = nn.Linear(out_dim, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.W(x)
        attn = self.leaky_relu(self.a_src(h) + self.a_dst(h).transpose(-2, -1))
        return torch.bmm(self.dropout(F.softmax(attn, dim=-1)), h)

class MultiHeadGraphAttention(nn.Module):
    def __init__(self, in_dim, out_dim, n_heads=2, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList([GraphAttentionLayer(in_dim, out_dim, dropout) for _ in range(n_heads)])
        self.norm = nn.LayerNorm(out_dim * n_heads)
    def forward(self, x):
        return self.norm(torch.cat([h(x) for h in self.heads], dim=-1))

class AASSISTBackend(nn.Module):
    """Full AASIST backend for teacher (hidden_dim=160)."""
    def __init__(self, input_dim=1024, hidden_dim=160, n_heads=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.GELU(),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout))
        self.spectral_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.spectral_pool = nn.AdaptiveAvgPool1d(1)
        self.temporal_gat = MultiHeadGraphAttention(hidden_dim, hidden_dim, n_heads, dropout)
        self.temporal_pool = nn.AdaptiveAvgPool1d(1)
        cd = hidden_dim * n_heads * 2
        self.hetero_attention = nn.Sequential(
            nn.Linear(cd, hidden_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, 2))

    def forward(self, features):
        x = self.projection(features)
        sp = self.spectral_pool(self.spectral_gat(x).transpose(1, 2)).squeeze(-1)
        tp = self.temporal_pool(self.temporal_gat(x).transpose(1, 2)).squeeze(-1)
        return self.classifier(self.hetero_attention(torch.cat([sp, tp], dim=-1)))

class TeacherModel(nn.Module):
    def __init__(self, w2v_name="facebook/wav2vec2-xls-r-300m", hidden_dim=160):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(w2v_name)
        self.wav2vec2.feature_extractor._freeze_parameters()
        self.backend = AASSISTBackend(
            input_dim=self.wav2vec2.config.hidden_size, hidden_dim=hidden_dim)

    def forward(self, audio, mask=None):
        if mask is not None:
            mask = mask.long()
        features = self.wav2vec2(audio, attention_mask=mask).last_hidden_state
        return self.backend(features)

    def get_param_groups(self):
        return [
            {"params": self.wav2vec2.parameters(), "lr": LEARNING_RATE_W2V},
            {"params": self.backend.parameters(), "lr": LEARNING_RATE_BACKEND},
        ]

In [ ]:
def count_params(model, trainable_only=True):
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())

print("Loading Wav2Vec2-XLSR (takes 1-2 min)...")
teacher = TeacherModel().to(DEVICE)
print(f"Total params:     {count_params(teacher, False):,}")
print(f"Trainable params: {count_params(teacher, True):,}")

# Quick test
with torch.no_grad():
    out = teacher(torch.randn(2, MAX_SAMPLES).to(DEVICE),
                  torch.ones(2, MAX_SAMPLES).to(DEVICE))
    print(f"Output shape: {out.shape}")  # [2, 2]
print("Model ready!")

## 3.4 Loss Function & Metrics

In [ ]:
class OCSoftmax(nn.Module):
    """
    OC-Softmax: adds margins to push bonafide and spoof predictions apart.
    Think of it as making the model more confident in its decisions.
    """
    def __init__(self, m_real=0.9, m_fake=0.2, alpha=20.0):
        super().__init__()
        self.m_real = m_real; self.m_fake = m_fake; self.alpha = alpha

    def forward(self, logits, labels):
        real_mask = (labels == 0).float()
        spoof_mask = (labels == 1).float()
        mod = logits.clone()
        mod[:, 0] = logits[:, 0] - self.m_real * real_mask
        mod[:, 1] = logits[:, 1] - self.m_fake * spoof_mask
        return F.cross_entropy(self.alpha * mod, labels)

def compute_eer(labels, scores):
    """Equal Error Rate: the point where false accepts = false rejects."""
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2, thresholds[idx]

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0; all_scores, all_preds, all_labels = [], [], []
    for batch in tqdm(dataloader, desc="Evaluating"):
        audio = batch["audio"].to(device); mask = batch["mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(audio, mask)
        total_loss += criterion(logits, labels).item()
        all_scores.extend(F.softmax(logits, dim=-1)[:, 1].cpu().numpy())
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    labels_arr = np.array(all_labels); scores_arr = np.array(all_scores)
    eer, _ = compute_eer(labels_arr, scores_arr)
    return {"loss": total_loss/len(dataloader), "eer": eer,
            "accuracy": accuracy_score(labels_arr, np.array(all_preds))}

## 3.5 Training Loop with Checkpoint Resume

**Checkpoint strategy:**
- After every epoch, the full state is saved to Google Drive
- If Colab disconnects, just run this notebook again
- It automatically detects the checkpoint and continues training

In [ ]:
criterion = OCSoftmax().to(DEVICE)
optimizer = torch.optim.AdamW(teacher.get_param_groups(), weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
# --- Check for existing checkpoint to resume from ---
TEACHER_CKPT_PATH = CHECKPOINT_DIR / "teacher_latest.pt"
TEACHER_BEST_PATH = CHECKPOINT_DIR / "teacher_best.pt"

start_epoch = 1
best_dev_eer = float("inf")
history = {"train_loss": [], "train_acc": [], "dev_eer": [], "dev_loss": []}

if TEACHER_CKPT_PATH.exists():
    print("Found existing checkpoint! Resuming training...")
    ckpt = torch.load(str(TEACHER_CKPT_PATH), map_location=DEVICE)
    teacher.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_dev_eer = ckpt["best_dev_eer"]
    history = ckpt["history"]
    print(f"Resuming from epoch {start_epoch} (best EER so far: {best_dev_eer*100:.2f}%)")
else:
    print("No checkpoint found. Starting fresh training.")

In [ ]:
print("=" * 60)
print(f"TEACHER TRAINING: Epochs {start_epoch} to {NUM_EPOCHS}")
print("=" * 60)

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    print(f"\n{'='*40} Epoch {epoch}/{NUM_EPOCHS} {'='*40}")

    # --- Train one epoch ---
    teacher.train()
    epoch_loss = 0; epoch_preds = []; epoch_labels = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")
    for batch in pbar:
        audio = batch["audio"].to(DEVICE)
        mask = batch["mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = teacher(audio, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(teacher.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        epoch_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
        epoch_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = epoch_loss / len(train_loader)
    train_acc = accuracy_score(epoch_labels, epoch_preds)

    # --- Validate ---
    dev_res = evaluate(teacher, dev_loader, criterion, DEVICE)
    scheduler.step()

    # --- Log ---
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["dev_eer"].append(dev_res["eer"])
    history["dev_loss"].append(dev_res["loss"])

    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Dev Loss:   {dev_res['loss']:.4f} | Dev EER: {dev_res['eer']*100:.2f}%")

    # --- Save best model ---
    if dev_res["eer"] < best_dev_eer:
        best_dev_eer = dev_res["eer"]
        torch.save({"epoch": epoch, "model_state_dict": teacher.state_dict(),
                     "dev_eer": best_dev_eer},
                   str(TEACHER_BEST_PATH))
        print(f"  ★ New best model saved! (EER: {best_dev_eer*100:.2f}%)")

    # --- Save checkpoint for resume (every epoch) ---
    torch.save({
        "epoch": epoch,
        "model_state_dict": teacher.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_dev_eer": best_dev_eer,
        "history": history,
    }, str(TEACHER_CKPT_PATH))
    print(f"  Checkpoint saved to Google Drive (epoch {epoch})")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE! Best Dev EER: {best_dev_eer*100:.2f}%")
print(f"{'='*60}")

## 3.6 Training Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history["train_loss"], "o-", label="Train")
axes[0].plot(history["dev_loss"], "o-", label="Dev")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history["train_acc"], "o-", color="green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Train Accuracy"); axes[1].grid(True, alpha=0.3)

axes[2].plot([e*100 for e in history["dev_eer"]], "o-", color="red")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("EER (%)")
axes[2].set_title("Dev EER"); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / "teacher_training_plots.png"), dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Notebook 3 complete. Proceed to Notebook 4: Knowledge Distillation.")